In [3]:
import sys
print(sys.executable)


C:\Users\deeks\AppData\Local\Programs\Python\Python313\python.exe


In [2]:
import torch


In [3]:
import os
import random
from typing import Literal, Optional, Tuple

import pandas as pd
from PIL import Image, ImageFilter
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn


In [11]:
DomainType = Literal["A", "B", "C", "all"]  # A=full, B=crop, C=background


class CUBDomainDataset(Dataset):
    """
    CUB dataset with synthetic domains:
      - Domain 0 (A): full image
      - Domain 1 (B): cropped bird
      - Domain 2 (C): background-focused (bird blurred)
    """

    def __init__(
        self,
        root: str,
        split: Literal["train", "test"] = "train",
        which_domain: DomainType = "all",
        transform: Optional[transforms.Compose] = None,
        blur_radius: int = 10,
    ):
        self.root = root
        self.images_folder = os.path.join(root, "images")
        self.which_domain = which_domain
        self.transform = transform
        self.blur_radius = blur_radius

        images_txt = os.path.join(root, "images.txt")
        labels_txt = os.path.join(root, "image_class_labels.txt")
        split_txt = os.path.join(root, "train_test_split.txt")
        bbox_txt = os.path.join(root, "bounding_boxes.txt")

        images_df = pd.read_csv(images_txt, sep=" ", header=None, names=["img_id", "img_path"])
        labels_df = pd.read_csv(labels_txt, sep=" ", header=None, names=["img_id", "class_id"])
        split_df  = pd.read_csv(split_txt,  sep=" ", header=None, names=["img_id", "is_train"])
        bbox_df   = pd.read_csv(bbox_txt,   sep=" ", header=None, names=["img_id", "x", "y", "w", "h"])

        bbox_df["x2"] = bbox_df["x"] + bbox_df["w"]
        bbox_df["y2"] = bbox_df["y"] + bbox_df["h"]

        df = (
            images_df
            .merge(labels_df, on="img_id")
            .merge(split_df,  on="img_id")
            .merge(bbox_df,   on="img_id")
        )

        if split == "train":
            df = df[df["is_train"] == 1]
        else:
            df = df[df["is_train"] == 0]

        df["class_id"] = df["class_id"] - 1  # 1..200 → 0..199
        self.df = df.reset_index(drop=True)

        if self.transform is None:
            self.transform = transforms.Compose(
                [
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                ]
            )

    def __len__(self):
        return len(self.df)

    def _load_full(self, row):
        img_rel = row["img_path"]
        img_path = os.path.join(self.images_folder, img_rel)
        return Image.open(img_path).convert("RGB")

    def _crop_bird(self, img, row):
        x1, y1, x2, y2 = int(row["x"]), int(row["y"]), int(row["x2"]), int(row["y2"])
        return img.crop((x1, y1, x2, y2))

    def _background_focused(self, img, row):
        x1, y1, x2, y2 = int(row["x"]), int(row["y"]), int(row["x2"]), int(row["y2"])
        bg = img.copy()
        bird = bg.crop((x1, y1, x2, y2)).filter(ImageFilter.GaussianBlur(self.blur_radius))
        bg.paste(bird, (x1, y1))
        return bg

    def __getitem__(self, idx) -> Tuple[torch.Tensor, int, int]:
        row = self.df.iloc[idx]
        label = int(row["class_id"])

        # decide which domain to use
        if self.which_domain == "A":
            domain_id = 0
        elif self.which_domain == "B":
            domain_id = 1
        elif self.which_domain == "C":
            domain_id = 2
        else:  # "all"
            domain_id = random.choice([0, 1, 2])

        full_img = self._load_full(row)
        if domain_id == 0:
            pil_img = full_img
        elif domain_id == 1:
            pil_img = self._crop_bird(full_img, row)
        else:
            pil_img = self._background_focused(full_img, row)

        img_tensor = self.transform(pil_img)
        return img_tensor, label, domain_id


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

root = r"C:\Users\deeks\.cache\kagglehub\datasets\wenewone\cub2002011\versions\7\CUB_200_2011" 

train_ds = CUBDomainDataset(root, split="train", which_domain="A")  # train on full images
test_A  = CUBDomainDataset(root, split="test", which_domain="A")
test_B  = CUBDomainDataset(root, split="test", which_domain="B")
test_C  = CUBDomainDataset(root, split="test", which_domain="C")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loaders = {
    "Full (A)": DataLoader(test_A, batch_size=64, shuffle=False),
    "Crop (B)": DataLoader(test_B, batch_size=64, shuffle=False),
    "Background (C)": DataLoader(test_C, batch_size=64, shuffle=False),
}

# Simple baseline model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200 classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [13]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels, _domain in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    correct, total = 0, 0

    for imgs, labels, _domain in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


In [14]:
num_epochs = 3  

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    print(f"Epoch {epoch}: loss={train_loss:.4f}, acc={train_acc:.3f}")


Epoch 1: loss=4.3461, acc=0.168
Epoch 2: loss=2.5648, acc=0.620
Epoch 3: loss=1.4741, acc=0.843


In [15]:
print("\nCross-Domain Accuracy (Train on A):\n")
for name, loader in test_loaders.items():
    acc = eval_model(model, loader)
    print(f"{name}: {acc:.3f}")



Cross-Domain Accuracy (Train on A):

Full (A): 0.628
Crop (B): 0.582
Background (C): 0.065
